In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from utils.silver_functions import *
# Lecture Bronze
# Les id sont changés dans la phase de silver_suppliers
incidents = spark.table("workspace.final_project.bronze_incidents")
incidents_quarantine = incidents.limit(0).withColumn("reason", lit(''))
display(incidents)
# Il faudra prendre les silvers quand ils seront prets.
suppliers = spark.table("workspace.final_project.bronze_suppliers")
orders = spark.table("workspace.final_project.bronze_orders")


In [0]:
# incident_type: normalisation string
incidents = incidents.withColumn(
    "incident_type",
    initcap(
        regexp_replace(
            regexp_replace(trim(col("incident_type")), "[^a-zA-Z0-9 ]", ""),
            "[àáâä]", "a"
        )
    )
)

In [0]:
# severity: normalisation string
incidents = incidents.withColumn(
    "severity",
    initcap(
        regexp_replace(trim(col("severity")), "[^a-zA-Z0-9 ]", "")
    )
)

In [0]:
# On traite order_id avant les autres car on va se servir pottentiellement de orders dans le traitement des autres colonnes 
# order_id: normalisation int + quanrantaine des nulls (incidents doit être rattaché a un order)
# + vérification présence dans la table bronze_orders 

incidents = incidents.withColumn(
    "order_id_parsed",
    col("order_id").try_cast(LongType())
)

# lignes invalides (format incorrect)
invalid_order_format = (
    incidents
    .filter(col("order_id_parsed").isNull())
    .withColumn("reason", lit("Invalid order_id format"))
    .drop("order_id_parsed")
)

incidents_quarantine = incidents_quarantine.unionByName(invalid_order_format)

# lignes valides
incidents = (
    incidents
    .filter(col("order_id_parsed").isNotNull())
    .drop("order_id")
    .withColumnRenamed("order_id_parsed", "order_id")
)


# order_id absent de la table orders
invalid_order_not_found = (
    incidents
    .join(
        orders.select("order_id"),
        on="order_id",
        how="left_anti"
    )
    .withColumn("reason", lit("order_id not found in orders table"))
)

incidents_quarantine = incidents_quarantine.unionByName(invalid_order_not_found)

# garder uniquement les order_id présents dans orders
incidents = incidents.join(
    orders.select("order_id"),
    on="order_id",
    how="inner"
)

In [0]:
# incident_date: normalisation date
# On ne supprime pas les date à null: laisser null ou remplacer par la date d'arriver ?
incidents = incidents.withColumn(
    "incident_date",
    coalesce(
        to_date("incident_date", "yyyy-MM-dd"),
        to_date("incident_date", "dd/MM/yyyy"),
        to_date("incident_date", "yyyy/MM/dd")
    )
)

In [0]:
# supplier_id: normalisation int + vérification présence dans la table bronze_suppliers.
# Si on ne trouve pas supplier_id on va le chercher dans la table orders

# normalisation supplier_id
incidents = incidents.withColumn(
    "supplier_id",
    incidents.supplier_id.try_cast(LongType())
)

# récupérer supplier_id depuis orders si absent
incidents = incidents.join(
    orders.select("order_id", "supplier_id").withColumnRenamed("supplier_id", "supplier_id_from_order"),
    on="order_id",
    how="left"
)

incidents = incidents.withColumn(
    "supplier_id",
    coalesce(col("supplier_id"), col("supplier_id_from_order"))
).drop("supplier_id_from_order")


# supplier_id toujours absent -> quarantaine
# ca ne devrait jamais arriver car supplier_id est présent dans toutes les lignes de la table orders
invalid_supplier_not_found = (
    incidents
    .join(
        suppliers.select("supplier_id"),
        on="supplier_id",
        how="left_anti"
    )
    .withColumn("reason", lit("supplier_id not found in suppliers or orders"))
)

incidents_quarantine = incidents_quarantine.unionByName(invalid_supplier_not_found)


# garder uniquement les supplier valides
incidents = incidents.join(
    suppliers.select("supplier_id"),
    on="supplier_id",
    how="inner"
)


In [0]:
incidents = incidents.withColumn(
    "description",
    trim(col("description"))
)

In [0]:
incidents.show()

In [0]:
incidents.write.mode("overwrite").saveAsTable(
    "workspace.final_project.silver_incidents"
)
incidents_quarantine.write.mode("overwrite").saveAsTable(
    "workspace.final_project.incidents_quarantine"
)